In [164]:
# Nepali IM Bullying detection using LSTM: Multi-label classification
# Labels:
#   1 -> Insult
#   2 -> Threat
#   3 -> Harassment
#   4 -> IdentityAttack

import re
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from collections import Counter
from torch.utils.data import Dataset, DataLoader
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,classification_report

In [165]:
SEED= 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)

Device: cpu


In [166]:
DATA_PATH=  "nepali_bullying_dataset_1000.csv"
df=  pd.read_csv(DATA_PATH)
df

,id,Context,Message,Insult,Threat,Harassment,IdentityAttack
0,1,School,तँ त कक्षामा सबैभन्दा बेकार विद्यार्थी होस्,1,0,0,0
1,2,School,"यस्तो सजिलो प्रश्नको जवाफ पनि थाहा छैन, कति मूर्ख",1,0,0,0
2,3,School,"तिम्रो हस्तलेखन हेर्दा हाँसो उठ्छ, कति बदमास ल...",1,0,0,0
3,4,School,तिमी त हरेक पटक फेल हुने विद्यार्थी होस्,1,0,0,0
4,5,School,"भोलि विद्यालय पछाडि भेट्, तिमीलाई कुट्नेछु",0,1,0,0
...,...,...,...,...,...,...,...
995,996,PublicInterest,"फेरि भन्छु, नयाँ नीतिले सर्वसाधारणलाई प्रत्यक्...",0,0,0,0
996,997,PublicInterest,"अरे, सामुदायिक कार्यक्रम सफलतापूर्वक सम्पन्न भ...",0,0,0,0
997,998,PublicInterest,"अरे, यो अभियानले धेरै जनाको ध्यान तान्न सफल भयो",0,0,0,0
998,999,PublicInterest,सार्वजनिक छलफलमा सहभागी हुनुभएकोमा धन्यवाद,0,0,0,0


In [167]:
df.isna().sum()

,0
id,0
Context,0
Message,0
Insult,0
Threat,0
Harassment,0
IdentityAttack,0


In [168]:
feature_cols = ['Message']
LABELS = df.columns[-4:].tolist()
# Number of labels
NUM_LABELS = len(LABELS)

# Convert labels to integers
df[LABELS] = df[LABELS].astype(int)

df.dtypes

,0
id,int64
Context,object
Message,object
Insult,int64
Threat,int64
Harassment,int64
IdentityAttack,int64


In [169]:
# Features and targets

X = df[feature_cols]
y = df[LABELS]

# --------------------------------------------------
# Step 1: Split into 70% Train and 30% Temporary
# --------------------------------------------------
ms1 = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=42
)
train_idx, temp_idx = next(ms1.split(X, y))

train_df = df.iloc[train_idx].copy()
temp_df = df.iloc[temp_idx].copy()

# --------------------------------------------------
# Step 2: Split Temporary into 15% Validation and 15% Test
# --------------------------------------------------
ms2 = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=42
)
val_idx, test_idx = next(
    ms2.split(temp_df[feature_cols], temp_df[LABELS])
)

val_df = temp_df.iloc[val_idx].copy()
test_df = temp_df.iloc[test_idx].copy()

# --------------------------------------------------
# Check dataset sizes
# --------------------------------------------------
print("Train      :", train_df.shape)
print("Validation :", val_df.shape)
print("Test       :", test_df.shape)

distribution = pd.DataFrame({
    "Train %": train_df[LABELS].mean() * 100,
    "Validation %": val_df[LABELS].mean() * 100,
    "Test %": test_df[LABELS].mean() * 100
})
print(distribution.round(2))

Train      : (700, 7)
Validation : (150, 7)
Test       : (150, 7)
                Train %  Validation %  Test %
Insult            23.00         22.67   23.33
Threat            21.71         21.33   22.00
Harassment        23.86         24.00   24.00
IdentityAttack    20.43         20.67   20.00


In [170]:
def tokenize(text):
    # Keep Nepali Unicode characters,
    # numbers and whitespace.
    # Devanagari range:\u0900-\u097F
    text = re.sub(r"[^\u0900-\u097F0-9\s]", " ", text)
    # Remove extra whitespace
    text = re.sub(r"\s+"," ",text).strip()
    return text.split()

In [171]:
counter = Counter()
for text in train_df["Message"]:
    tokens = tokenize(text)
    counter.update(tokens)
# Special tokens
PAD_TOKEN = "<PAD>"
UNK_TOKEN  ="<UNK>"
word_to_idx = {PAD_TOKEN: 0, UNK_TOKEN: 1}
# Minimum frequency
MIN_FREQ = 2
for word, frequency in counter.most_common():
    if frequency >= MIN_FREQ:
        word_to_idx[word] = len(word_to_idx)
idx_to_word = {idx: word for word, idx in word_to_idx.items()}

VOCAB_SIZE = len(word_to_idx)
print("\nVocabulary size:", VOCAB_SIZE)


Vocabulary size: 669


In [172]:
def encode_text(text):
    tokens = tokenize(text)
    if not tokens:
        tokens = [UNK_TOKEN]
    return [word_to_idx.get(t, word_to_idx[UNK_TOKEN]) for t in tokens]

In [173]:
class NewsBiasDataset(Dataset):

    def __init__(self, dataframe):

        self.texts = dataframe["Message"].tolist()

        self.labels = dataframe[LABELS].values.astype(np.float32)

    def __len__(self):

        return len(self.texts)

    def __getitem__(self, index):

        text = self.texts[index]

        label = self.labels[index]

        encoded = encode_text(text)

        return (
            torch.tensor(
                encoded,
                dtype=torch.long
            ),
            torch.tensor(
                label,
                dtype=torch.float
            )
        )

train_dataset = NewsBiasDataset(train_df)
val_dataset = NewsBiasDataset(val_df)
test_dataset = NewsBiasDataset(test_df)

In [174]:
# Headlines have different lengths.
def collate_fn(batch):

    sequences, labels = zip(*batch)

    lengths = torch.tensor(
        [len(seq) for seq in sequences],
        dtype=torch.long
    )

    # Prevent zero-length sequences

    lengths = torch.clamp(lengths,min=1)

    padded_sequences = nn.utils.rnn.pad_sequence(sequences,batch_first=True,padding_value=0)

    labels = torch.stack(labels)

    return (
        padded_sequences,
        lengths,
        labels
    )

In [175]:
BATCH_SIZE = 64
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

In [176]:
class NewsBiasLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=128,
                 num_layers=2, dropout=0.3, num_labels=4):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.dropout = nn.Dropout(dropout)
        # bidirectional (x2) + mean&max pooling concat (x2)
        self.fc = nn.Linear(hidden_dim * 2 * 2, num_labels)

    def forward(self, x, lengths):
        embedded = self.embedding(x)
        packed = nn.utils.rnn.pack_padded_sequence(
            embedded, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        packed_output, _ = self.lstm(packed)
        output, _ = nn.utils.rnn.pad_packed_sequence(packed_output, batch_first=True)

        mask = (x != 0).unsqueeze(-1).float()
        mean_pool = (output * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
        max_pool = output.masked_fill(mask == 0, -1e9).max(1).values

        combined = torch.cat([mean_pool, max_pool], dim=1)
        combined = self.dropout(combined)
        return self.fc(combined)

In [177]:
model = NewsBiasLSTM(
    vocab_size=VOCAB_SIZE,
    embedding_dim=128,
    hidden_dim=128,
    num_layers=2,
    dropout=0.3,
    num_labels=NUM_LABELS
)

model = model.to(DEVICE)

print("\nModel:")
print(model)


Model:
NewsBiasLSTM(
  (embedding): Embedding(669, 128, padding_idx=0)
  (lstm): LSTM(128, 128, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=512, out_features=4, bias=True)
)


In [178]:
pos_counts = train_df[LABELS].sum().values
neg_counts = len(train_df) - pos_counts
pos_weight = torch.tensor(neg_counts / pos_counts, dtype=torch.float32).to(DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

In [179]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)


# 16. Evaluation Function
def evaluate(
    model,
    data_loader,
    threshold=0.5
):

    model.eval()

    all_predictions = []
    all_labels = []

    total_loss = 0.0

    with torch.no_grad():

        for x, lengths, labels in data_loader:

            # Move data to device
            x = x.to(DEVICE)
            lengths = lengths.to(DEVICE)
            labels = labels.to(DEVICE)

            # Forward pass


            logits = model(
                x,
                lengths
            )


            # Calculate loss


            loss = criterion(
                logits,
                labels
            )

            total_loss += loss.item()

            # Convert logits → probabilities

            probabilities = torch.sigmoid(
                logits
            )

            # Probability → binary prediction

            predictions = (
                probabilities >= threshold
            ).int()


            # Store predictions and labels

            all_predictions.append(
                predictions.cpu().numpy()
            )

            all_labels.append(
                labels.cpu().numpy()
            )


    # Combine batches

    all_predictions = np.vstack(
        all_predictions
    )

    all_labels = np.vstack(
        all_labels
    )

      # Average loss

    avg_loss = (
        total_loss / len(data_loader)
    )

     # Metrics


    micro_f1 = f1_score(
        all_labels,
        all_predictions,
        average="micro",
        zero_division=0
    )

    macro_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )

    micro_precision = precision_score(
        all_labels,
        all_predictions,
        average="micro",
        zero_division=0
    )

    micro_recall = recall_score(
        all_labels,
        all_predictions,
        average="micro",
        zero_division=0
    )
    # Return results

    return {
        "loss": avg_loss,
        "micro_f1": micro_f1,
        "macro_f1": macro_f1,
        "precision": micro_precision,
        "recall": micro_recall,
        "predictions": all_predictions,
        "labels": all_labels
    }

In [180]:
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', patience=2, factor=0.5
)

EPOCHS = 10
PATIENCE = 4
best_val_f1 = 0.0
epochs_no_improve = 0

for epoch in range(EPOCHS):
    model.train()
    total_train_loss = 0.0

    for x, lengths, labels in train_loader:
        x, lengths, labels = x.to(DEVICE), lengths.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(x, lengths)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader)
    val_results = evaluate(model, val_loader)
    scheduler.step(val_results["micro_f1"])

    print(f"\nEpoch {epoch + 1}/{EPOCHS}")
    print(f"Train Loss: {avg_train_loss:.4f}")
    print(f"Val Loss:   {val_results['loss']:.4f}")
    print(f"Val F1:     {val_results['micro_f1']:.4f}")
    print(f"Val Macro F1: {val_results['macro_f1']:.4f}")

    if val_results["micro_f1"] > best_val_f1:
        best_val_f1 = val_results["micro_f1"]
        epochs_no_improve = 0
        torch.save(model.state_dict(), "best_lstm_model.pt")
        print("Best model saved.")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"Early stopping at epoch {epoch + 1}")
            break


Epoch 1/10
Train Loss: 1.0473
Val Loss:   0.9786
Val F1:     0.6404
Val Macro F1: 0.6764
Best model saved.

Epoch 2/10
Train Loss: 0.8460
Val Loss:   0.6512
Val F1:     0.7293
Val Macro F1: 0.7347
Best model saved.

Epoch 3/10
Train Loss: 0.4381
Val Loss:   0.2773
Val F1:     0.8767
Val Macro F1: 0.8807
Best model saved.

Epoch 4/10
Train Loss: 0.1497
Val Loss:   0.0770
Val F1:     0.9635
Val Macro F1: 0.9644
Best model saved.

Epoch 5/10
Train Loss: 0.0534
Val Loss:   0.0281
Val F1:     0.9925
Val Macro F1: 0.9924
Best model saved.

Epoch 6/10
Train Loss: 0.0200
Val Loss:   0.0123
Val F1:     0.9925
Val Macro F1: 0.9924

Epoch 7/10
Train Loss: 0.0094
Val Loss:   0.0076
Val F1:     1.0000
Val Macro F1: 1.0000
Best model saved.

Epoch 8/10
Train Loss: 0.0056
Val Loss:   0.0058
Val F1:     1.0000
Val Macro F1: 1.0000

Epoch 9/10
Train Loss: 0.0041
Val Loss:   0.0048
Val F1:     1.0000
Val Macro F1: 1.0000

Epoch 10/10
Train Loss: 0.0033
Val Loss:   0.0041
Val F1:     1.0000
Val Macro F1

In [181]:
model.load_state_dict(
    torch.load("best_lstm_model.pt", map_location=DEVICE)
)
model.eval()

NewsBiasLSTM(
  (embedding): Embedding(669, 128, padding_idx=0)
  (lstm): LSTM(128, 128, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=512, out_features=4, bias=True)
)

In [182]:
test_results = evaluate(model, test_loader)

print("TEST RESULTS")
print(f"Loss:      {test_results['loss']:.4f}")
print(f"Precision: {test_results['precision']:.4f}")
print(f"Recall:    {test_results['recall']:.4f}")
print(f"Micro F1:  {test_results['micro_f1']:.4f}")
print(f"Macro F1:  {test_results['macro_f1']:.4f}")

TEST RESULTS
Loss:      0.0077
Precision: 0.9926
Recall:    1.0000
Micro F1:  0.9963
Macro F1:  0.9963


In [183]:
print("\nClassification Report:\n")
print(classification_report(
    test_results["labels"],
    test_results["predictions"],
    target_names=LABELS,
    zero_division=0
))


Classification Report:

                precision    recall  f1-score   support

        Insult       1.00      1.00      1.00        35
        Threat       0.97      1.00      0.99        33
    Harassment       1.00      1.00      1.00        36
IdentityAttack       1.00      1.00      1.00        30

     micro avg       0.99      1.00      1.00       134
     macro avg       0.99      1.00      1.00       134
  weighted avg       0.99      1.00      1.00       134
   samples avg       0.72      0.72      0.72       134



In [184]:
def predict_bully(messages, threshold=0.5):
    model.eval()

    ids = encode_text(messages)
    x = torch.tensor([ids], dtype=torch.long).to(DEVICE)
    lengths = torch.tensor([len(ids)], dtype=torch.long).to(DEVICE)

    with torch.no_grad():
        logits = model(x, lengths)
        probabilities = torch.sigmoid(logits)[0].cpu().numpy()

    return {
        label: {"probability": float(prob), "predicted": bool(prob >= threshold)}
        for label, prob in zip(LABELS, probabilities)
    }

In [187]:
message= "तिमीसँग बोल्नु भन्दा बरु कुकुरसँग बोल्छुँ;कति राम्रो कुकुर ।"
results = predict_bully(message)

print("New Message:")
print(message)
print("\nPredictions:")
for label, info in results.items():
    print(f"{label:20s}: {info['probability']:.4f}")

New Message:
तिमीसँग बोल्नु भन्दा बरु कुकुरसँग बोल्छुँ;कति राम्रो कुकुर ।

Predictions:
Insult              : 0.0004
Threat              : 0.0133
Harassment          : 0.0025
IdentityAttack      : 0.0004


In [188]:
print("\nDetected Bullying types:")
detected = [label for label, info in results.items() if info["predicted"]]

if detected:
    for label in detected:
        print("-", label)
else:
    print("No Bullying type detected.")


Detected Bullying types:
No Bullying type detected.
